In [ ]:
from IPython.display import Markdown
import pyaudio
import wave
import keyboard
import threading
import time
import openai
import os
from datetime import datetime

class AudioTranscriber:
    def __init__(self):
        self.CHUNK = 1024
        self.FORMAT = pyaudio.paInt16
        self.CHANNELS = 1
        self.RATE = 44100
        self.recording = False
        self.frames = []
        
    def start_recording(self):
        """Start recording audio from microphone"""
        self.recording = True
        self.frames = []
        p = pyaudio.PyAudio()
        
        stream = p.open(format=self.FORMAT,
                       channels=self.CHANNELS,
                       rate=self.RATE,
                       input=True,
                       frames_per_buffer=self.CHUNK)
        
        print("Recording started... Press 'q' to stop recording")
        
        while self.recording:
            data = stream.read(self.CHUNK)
            self.frames.append(data)
            
        stream.stop_stream()
        stream.close()
        p.terminate()
        
    def save_audio(self, filename):
        """Save recorded audio to WAV file"""
        p = pyaudio.PyAudio()
        wf = wave.open(filename, 'wb')
        wf.setnchannels(self.CHANNELS)
        wf.setsampwidth(p.get_sample_size(self.FORMAT))
        wf.setframerate(self.RATE)
        wf.writeframes(b''.join(self.frames))
        wf.close()
        
    def transcribe_audio(self, audio_file):
        """Transcribe audio file using OpenAI Whisper API"""
        try:
            with open(audio_file, "rb") as file:
                transcript = openai.audio.transcriptions.create(
                    model="whisper-1",
                    file=file
                )
            return transcript.text
        except Exception as e:
            print(f"Error during transcription: {str(e)}")
            return None

    def save_notes(self, text, filename):
        """Save transcribed text to a document"""
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(f"Notes from recording on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                f.write(text)
            print(f"Notes saved to {filename}")
        except Exception as e:
            print(f"Error saving notes: {str(e)}")

def main():
    # Create output directories if they don't exist
    if not os.path.exists('recordings'):
        os.makedirs('recordings')
    if not os.path.exists('transcripts'):
        os.makedirs('transcripts')
    
    transcriber = AudioTranscriber()
    
    def stop_recording():
        """Monitor for 'q' key press to stop recording"""
        keyboard.wait('q')
        transcriber.recording = False
    
    try:
        # Start recording in a separate thread
        recording_thread = threading.Thread(target=transcriber.start_recording)
        recording_thread.start()
        
        # Start monitoring for stop key in another thread
        stop_thread = threading.Thread(target=stop_recording)
        stop_thread.start()
        
        # Wait for recording to finish
        recording_thread.join()
        
        # Generate filenames with timestamp
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        audio_file = f'recordings/recording_{timestamp}.wav'
        transcript_file = f'transcripts/transcript_{timestamp}.txt'
        
        # Save the audio file
        transcriber.save_audio(audio_file)
        print(f"Audio saved to {audio_file}")
        
        # Transcribe the audio
        print("Transcribing audio...")
        transcribed_text = transcriber.transcribe_audio(audio_file)
        print(transcribed_text )
        
        if transcribed_text:
            # Save the transcription
            transcriber.save_notes(transcribed_text, transcript_file)
        else:
            print("Transcription failed")
            
    except Exception as e:
        print(f"An error occurred: {str(e)}")

    user_prompt=f"Read the transcript and organize the text into notes with points. Transcript is as follows + {transcribed_text}"
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "you are a AI assistant which read file in folder named transcripts. Also organized the transcript into notes with points "},
            {"role": "user", "content": user_prompt}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

if __name__ == "__main__":
    main()